In [51]:
import os
import requests
import json

from pathlib import Path
from datasets import load_dataset

import sys
sys.path.append("..")

In [52]:
data_path = Path("../data")
ckpt_path = Path("../ckpt")

ckpt_path.mkdir(exist_ok=True, parents=True)

warandpeace_path = data_path / "warandpeace.txt"


tokenizer_path = ckpt_path / 'tokenizer'
tokenizer_path.mkdir(exist_ok=True, parents=True)

In [53]:
with open(warandpeace_path, 'r') as file:
    warandpeace = file.read()

In [54]:
chars = sorted(list(set(warandpeace)))
vocab_size = len(chars)

print("".join(chars))
print(vocab_size)


 !"'()*,-.0123456789:;<=>?ABCDEFGHIJKLMNOPRSTUVX\^abcdefghilmnoprstuvwxyАБВГДЕЖЗИЙКЛМНОПРСТУФХЦЧШЩЬЭЮЯабвгдежзийклмнопрстуфхцчшщъыьэюяё
136


In [55]:
# split text to tokens


In [56]:
text = warandpeace[:1000]
print(text)

Лев Николаевич Толстой
Война и мир

Первый вариант романа

От автора

  
   Я пишу до сих пор только о князьях, графах, министрах, сенаторах и их детях и боюсь, что и вперед не будет других лиц в моей истории.
   Может быть, это нехорошо и не нравится публике; может быть, для нее интереснее и поучительнее история мужиков, купцов, семинаристов, но, со всем моим желанием иметь как можно больше читателей, я не могу угодить такому вкусу, по многим причинам.
   Во-первых, потому, что памятники истории того времени, о котором я пишу, остались только в переписке и записках людей высшего круга грамотных; даже интересные и умные рассказы, которые мне удалось слышать, слышал я только от людей того же круга.
   Во-вторых, потому, что жизнь купцов, кучеров, семинаристов, каторжников и мужиков для меня представляется однообразною и скучною, и все действия этих людей мне представляются вытекающими, большей частью, из одних и тех же пружин: зависти к более счастливым сословиям, корыстолюбия и материа

In [57]:
import re


splits = re.split(r"([,.:;?!-\"'()]|--|\d|\s)", text)
splits = [word for word in splits if word.strip()]
splits

['Лев',
 'Николаевич',
 'Толстой',
 'Война',
 'и',
 'мир',
 'Первый',
 'вариант',
 'романа',
 'От',
 'автора',
 'Я',
 'пишу',
 'до',
 'сих',
 'пор',
 'только',
 'о',
 'князьях',
 ',',
 'графах',
 ',',
 'министрах',
 ',',
 'сенаторах',
 'и',
 'их',
 'детях',
 'и',
 'боюсь',
 ',',
 'что',
 'и',
 'вперед',
 'не',
 'будет',
 'других',
 'лиц',
 'в',
 'моей',
 'истории',
 '.',
 'Может',
 'быть',
 ',',
 'это',
 'нехорошо',
 'и',
 'не',
 'нравится',
 'публике',
 ';',
 'может',
 'быть',
 ',',
 'для',
 'нее',
 'интереснее',
 'и',
 'поучительнее',
 'история',
 'мужиков',
 ',',
 'купцов',
 ',',
 'семинаристов',
 ',',
 'но',
 ',',
 'со',
 'всем',
 'моим',
 'желанием',
 'иметь',
 'как',
 'можно',
 'больше',
 'читателей',
 ',',
 'я',
 'не',
 'могу',
 'угодить',
 'такому',
 'вкусу',
 ',',
 'по',
 'многим',
 'причинам',
 '.',
 'Во-первых',
 ',',
 'потому',
 ',',
 'что',
 'памятники',
 'истории',
 'того',
 'времени',
 ',',
 'о',
 'котором',
 'я',
 'пишу',
 ',',
 'остались',
 'только',
 'в',
 'переписке'

In [58]:

tokenized = re.split(r"([,.:;?!-\"'()]|--|\d|\s)", warandpeace)
tokenized = [word for word in tokenized if word.strip()]
print("Количество слов в тексте", len(tokenized))
all_words = sorted(list(set(tokenized)))
print("Количество уникальных слов в тексте", len(all_words))
all_words[:15]

Количество слов в тексте 371024
Количество уникальных слов в тексте 40463


['!',
 '"',
 "'",
 '(',
 ')',
 '*',
 ',',
 '-',
 '--',
 '-верстной',
 '-го',
 '-е',
 '-й',
 '-летнего',
 '-летней']

In [59]:
all_words.extend(["<endoftext>", "<unk>"]) 
# <bos> - begining of sequence
# <eos> - end of sequence
# <pad> - padding

vocab = {token:i for i, token in enumerate(all_words)}

In [60]:

class SimpleTokenizer:

    def __init__(self, vocab, unk_token="<unk>"):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s, i in vocab.items()}
        self.unk_token = unk_token

    def encode(self, text):
        preprocessed = re.split(r"([,.:;?!-\"'()]|--|\d|\s)", text)
        preprocessed = [word if word in self.str_to_int else self.unk_token  for word in preprocessed if word.strip()]
        ids = [self.str_to_int[s] for s in preprocessed]

        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r"\s+([].,?!\"'])", r"\1", text)
        return text

In [61]:
tokenizer = SimpleTokenizer(vocab)

In [62]:
text = "эй йо привет мазафака, mon cher ой я пошел спать"
print(tokenizer.encode(text))
print(tokenizer.decode(tokenizer.encode(text)))


[40464, 40464, 40464, 40464, 6, 147, 127, 40464, 40355, 27457, 34653]
<unk> <unk> <unk> <unk>, mon cher <unk> я пошел спать


In [63]:
# Byte Pair Encoding from tiktoken

import tiktoken

In [64]:
tokenizer = tiktoken.get_encoding("gpt2")

In [65]:
encoded = tokenizer.encode(text + "<endoftext>", allowed_special={"<endoftext>"})
print(encoded)
print(tokenizer.decode(encoded))

[141, 235, 140, 117, 12466, 117, 15166, 12466, 123, 21169, 18849, 38857, 16843, 20375, 12466, 120, 16142, 140, 115, 16142, 141, 226, 16142, 31583, 16142, 11, 937, 21382, 12466, 122, 140, 117, 220, 40623, 12466, 123, 15166, 141, 230, 16843, 30143, 220, 21727, 140, 123, 16142, 20375, 45367, 27, 437, 1659, 5239, 29]
эй йо привет мазафака, mon cher ой я пошел спать<endoftext>


In [66]:
tokenizer.n_vocab

50257